# Charles Bridge Image Analysis Testing

Testing the composed prompt (shared template + public walking space type + Charles Bridge place filler) for CCTV image analysis using GPT-4 Vision API.

In [9]:
from cctv.utils.azure import load_azure_openai_config
from cctv.utils.paths import place_data_dir
from cctv.utils.prompts import load_place_prompt
from cctv.analysis.azure_vision import analyze_images

azure = load_azure_openai_config()
if not azure.is_configured:
    print("Missing Azure OpenAI configuration in .env file")
    print("Required: AZURE_OPENAI_API_KEY, AZURE_OPENAI_ENDPOINT, AZURE_OPENAI_MODEL")
else:
    print("Azure OpenAI configuration loaded")
    print(f"Endpoint: {azure.endpoint}")
    print(f"Model: {azure.model}")
    print(f"Request URL: {azure.api_url}")

✅ Azure OpenAI configuration loaded
🔗 Endpoint: https://openaiendpoint-uk-1.openai.azure.com/
📋 Model: gpt-4.1
📅 API Version: 2025-04-01-preview


In [10]:
charles_bridge_prompt_content = load_place_prompt("charles_bridge")
print("Assembled Charles Bridge prompt from template + type + place fillers")
print(charles_bridge_prompt_content[:400] + "...")

✅ Charles Bridge prompt loaded successfully
📋 Loaded structured prompt from file
🔄 Using markdown prompt directly for API calls


In [ ]:
print("Vision helpers imported from cctv.analysis.azure_vision")


In [ ]:
import json
from pathlib import Path

image_dir = place_data_dir("charles_bridge")
dataset = json.loads((image_dir / "annotations.json").read_text(encoding="utf-8"))
annotations = dataset.get("annotations", [])
annotations


In [ ]:
print("Use analyze_images(image_paths, prompt, config=azure) for one or more frames")


In [ ]:
print("Using assembled Charles Bridge prompt for multi-image analysis")


In [ ]:
print(f"Found {len(annotations)} annotated image pairs")

for i, annotation in enumerate(annotations):
    image_names = annotation["image_names"]
    expected_crowdedness = annotation["crowdedness_score"]

    print(f"\nTesting pair {i+1}/{len(annotations)}:")
    print(f"  Images: {image_names}")
    print(f"  Expected crowdedness: {expected_crowdedness}/10")

    image_paths = [str(image_dir / img_name) for img_name in image_names]
    missing_images = [path for path in image_paths if not Path(path).exists()]
    if missing_images:
        print(f"  Missing images: {[Path(p).name for p in missing_images]}")
        continue

    result_pair = analyze_images(image_paths, charles_bridge_prompt_content, config=azure)

    if "error" in result_pair:
        print(f"  Analysis error: {result_pair['error']}")
        continue

    print("  Analysis completed")
    analysis = result_pair.get("analysis")
    if not isinstance(analysis, dict):
        continue
    predicted_crowdedness = analysis.get("overcrowdedness_level")
    if predicted_crowdedness is None:
        continue
    difference = abs(predicted_crowdedness - expected_crowdedness)
    print(f"  Predicted crowdedness: {predicted_crowdedness}/10")
    print(f"  Difference: {difference} (Expected: {expected_crowdedness})")
    print(f"  Weather: {analysis.get('weather')}")
    print(f"  Time: {analysis.get('time_of_day')}")
    if difference <= 1:
        print("  Within 1 point of expected")
    elif difference <= 2:
        print("  Within 2 points of expected")
    else:
        print("  Off by more than 2 points")

print(f"\nTesting completed for {len(annotations)} annotated pairs")
